# Cosmos DB Live Container Migration - Parameterized (Fabric)

This notebook performs a **live migration** of a single Cosmos DB container using parameters passed from a parent notebook. It is designed to be called by `04_CosmosDB_Parallel_Container_Migration` for multi-container scenarios.

Converted from [CosmosDBLiveContainerMigration.scala](https://github.com/Azure/azure-sdk-for-java/tree/main/sdk/cosmos/azure-cosmos-spark_3/Samples/DatabricksLiveContainerMigration).

**How it works:**
| Step | Description |
|------|-------------|
| 1 | Receive parameters (endpoint, keys, database, container, etc.) |
| 2 | Configure Spark Cosmos DB catalog |
| 3 | Create target database, throughput control table, and target container |
| 4 | Initialize change feed reader and writer |
| 5 | Start streaming migration |

> **Note:** This notebook is called via `notebookutils.notebook.run()` or `runMultiple()`. Parameters are injected into the parameter cell below.

## Step 1: Configure Spark Session with Cosmos DB Spark Connector

In [1]:
%%configure -f
{
    "defaultLakehouse": {
        "name": "Cosmos_Migration"
    }
}

StatementMeta(, d44d3ec3-4eba-4ca1-85fa-40aa2de2be4a, -1, Finished, Available, Finished, True)

In [2]:
# JAR Loading: SKIPPED - Using Fabric Environment
# The Fabric Environment "CosmosDB_Migration_Env" already provides:
#   /usr/lib/library-manager/bin/libraries/scala/azure-cosmos-spark_3-5_2-12-4.41.0.jar
# Attempting to add it again causes "already registered" errors.
print("✅ Using Cosmos Spark JAR from Fabric Environment")
print("   Path: /usr/lib/library-manager/bin/libraries/scala/azure-cosmos-spark_3-5_2-12-4.41.0.jar")

StatementMeta(, d44d3ec3-4eba-4ca1-85fa-40aa2de2be4a, 5, Finished, Available, Finished, False)

✅ Using Cosmos Spark JAR from Fabric Environment
   Path: /usr/lib/library-manager/bin/libraries/scala/azure-cosmos-spark_3-5_2-12-4.41.0.jar


## Step 2: Parameters (Injected by Parent Notebook)

These default values are overridden when this notebook is called via `notebookutils.notebook.run()` with arguments.

> **Important:** Mark this cell as a **parameter cell** in the Fabric notebook UI so arguments from the parent notebook are injected correctly.

In [ ]:
# =============================================================================
# PARAMETER CELL — Mark this cell as "parameters" in the notebook UI
# These defaults are overridden when called from 04_CosmosDB_Parallel_Container_Migration
# =============================================================================

# Source config
cosmos_source_endpoint = "https://sourcedata.documents.azure.com:443/"  # Cosmos DB Account URI of the source account
cosmos_source_masterkey = ""  # Cosmos DB Account PRIMARY KEY of the source account
cosmos_source_database_name = "SampleDB"  # name of your source database
cosmos_source_container_name = "SampleContainer"  # name of the container you want to migrate
cosmos_source_throughput_control = "0.95"  # target percentage of available throughput for migration

# Target config
cosmos_target_endpoint = "https://targetdata.documents.azure.com:443/"  # Cosmos DB Account URI of the target account
cosmos_target_masterkey = ""  # Cosmos DB Account PRIMARY KEY of the target account
cosmos_target_database_name = "SampleDB"  # name of your target database
cosmos_target_container_name = "SampleContainer5"  # name of the target container
cosmos_target_container_partition_key = "/categoryId"  # partition key field for the target container (keep forward slash)
cosmos_target_container_provisioned_throughput = "10000"  # provisioned throughput for the target container

# Common config
cosmos_region = "[East US]"
clear_checkpoint = "false"  # set to "true" only for a full re-migration from scratch

## Step 3: Configure Cosmos DB Spark Catalog

In [ ]:
# Configure SOURCE Cosmos DB Spark Catalog
spark.conf.set("spark.sql.catalog.cosmosCatalogSource", "com.azure.cosmos.spark.CosmosCatalog")
spark.conf.set("spark.sql.catalog.cosmosCatalogSource.spark.cosmos.accountEndpoint", cosmos_source_endpoint)
spark.conf.set("spark.sql.catalog.cosmosCatalogSource.spark.cosmos.accountKey", cosmos_source_masterkey)

# Configure TARGET Cosmos DB Spark Catalog
spark.conf.set("spark.sql.catalog.cosmosCatalogTarget", "com.azure.cosmos.spark.CosmosCatalog")
spark.conf.set("spark.sql.catalog.cosmosCatalogTarget.spark.cosmos.accountEndpoint", cosmos_target_endpoint)
spark.conf.set("spark.sql.catalog.cosmosCatalogTarget.spark.cosmos.accountKey", cosmos_target_masterkey)

print(f"SOURCE Cosmos DB: {cosmos_source_endpoint}")
print(f"  Database/Container: {cosmos_source_database_name}/{cosmos_source_container_name}")
print(f"TARGET Cosmos DB: {cosmos_target_endpoint}")
print(f"  Database/Container: {cosmos_target_database_name}/{cosmos_target_container_name}")

In [ ]:
# SKIP driver-side verification - doesn't work in Fabric child notebooks
# The JAR is loaded on executors where Cosmos operations actually run
# If Cosmos operations fail, we'll get better error messages than this check provides
print("⏭️ Skipping JAR verification (Fabric child notebooks limitation)")
print("   JAR was added to Spark executors - proceeding with migration")
print("   If Cosmos operations fail, check that JAR downloaded successfully")

## Step 4: Create Target Database, Throughput Control & Target Container

- Creates the **target database** if it doesn't exist (handles shared throughput databases)
- Creates a **ThroughputControl** table in the source database for rate-limiting
- Creates the **target container** with specified partition key and throughput

In [ ]:
def get_database_properties(provisioned_throughput):
    """Returns DBPROPERTIES clause for shared throughput databases."""
    if "shared" in provisioned_throughput.lower():
        throughput_value = provisioned_throughput.replace("sharedDB", "").strip()
        return f"WITH DBPROPERTIES (autoScaleMaxThroughput = '{throughput_value}')"
    return ""

def get_container_throughput_clause(provisioned_throughput):
    """Returns throughput clause for dedicated container throughput."""
    if "shared" not in provisioned_throughput.lower():
        return f"autoScaleMaxThroughput = '{provisioned_throughput}',"
    return ""

# --- Create Target Database (IF NOT EXISTS) ---
db_properties = get_database_properties(cosmos_target_container_provisioned_throughput)
create_db_query = f"CREATE DATABASE IF NOT EXISTS cosmosCatalogTarget.`{cosmos_target_database_name}` {db_properties};"
print(f"✓ Creating target database: {cosmos_target_database_name}")
try:
    spark.sql(create_db_query)
    print(f"  ✅ Database '{cosmos_target_database_name}' created/verified in target")
except Exception as e:
    print(f"  ⚠️ Database creation issue: {str(e)[:100]}")
    # Continue anyway - database might already exist

# --- Create Throughput Control Table (IF NOT EXISTS) ---
create_tc_query = f"""
CREATE TABLE IF NOT EXISTS cosmosCatalogSource.`{cosmos_source_database_name}`.`ThroughputControl`
USING cosmos.oltp
OPTIONS (spark.cosmos.database = '{cosmos_source_database_name}')
TBLPROPERTIES(
    partitionKeyPath = '/groupId',
    autoScaleMaxThroughput = '4000',
    indexingPolicy = 'AllProperties',
    defaultTtlInSeconds = '-1'
)
"""
print(f"✓ Creating throughput control table in source database...")
try:
    spark.sql(create_tc_query)
    print(f"  ✅ ThroughputControl table created/verified in source")
except Exception as e:
    print(f"  ⚠️ ThroughputControl creation issue: {str(e)[:100]}")
    # Continue anyway - table might already exist

# --- Create Target Container (IF NOT EXISTS) ---
container_throughput = get_container_throughput_clause(cosmos_target_container_provisioned_throughput)
create_container_query = f"""
CREATE TABLE IF NOT EXISTS cosmosCatalogTarget.`{cosmos_target_database_name}`.`{cosmos_target_container_name}`
USING cosmos.oltp
TBLPROPERTIES(
    partitionKeyPath = '{cosmos_target_container_partition_key}',
    {container_throughput}
    indexingPolicy = 'OnlySystemProperties'
)
"""
print(f"✓ Creating target container: {cosmos_target_database_name}/{cosmos_target_container_name}")
print(f"  Partition key: {cosmos_target_container_partition_key}")
try:
    spark.sql(create_container_query)
    print(f"  ✅ Container '{cosmos_target_container_name}' created/verified in target")
except Exception as e:
    print(f"  ❌ Container creation FAILED: {str(e)[:200]}")
    raise Exception(f"Cannot proceed - target container creation failed: {e}")

## Step 5: Initialize Change Feed Reader & Start Streaming Write

Reads from the source container's change feed (from beginning) and writes to the target container. Checkpoint location is stored in the Lakehouse Files area, per source database/container, enabling resume after failures.

In [ ]:
# Change feed reader configuration (SOURCE)
# Ensure cosmos_region is defined (fallback if parameter cell didn't set it)
if 'cosmos_region' not in dir():
    cosmos_region = "[East US]"

change_feed_cfg = {
    "spark.cosmos.accountEndpoint": cosmos_source_endpoint,
    "spark.cosmos.accountKey": cosmos_source_masterkey,
    "spark.cosmos.applicationName": f"{cosmos_source_database_name}_{cosmos_source_container_name}_LiveMigrationRead_",
    "spark.cosmos.database": cosmos_source_database_name,
    "spark.cosmos.container": cosmos_source_container_name,
    "spark.cosmos.read.partitioning.strategy": "Restrictive",
    "spark.cosmos.read.inferSchema.enabled": "false",
    "spark.cosmos.read.maxItemCount": "5",
    "spark.cosmos.changeFeed.startFrom": "Beginning",
    "spark.cosmos.changeFeed.mode": "Incremental",
    "spark.cosmos.changeFeed.itemCountPerTriggerHint": "50000",
    # Throughput control
    "spark.cosmos.throughputControl.enabled": "true",
    "spark.cosmos.throughputControl.name": "SourceContainerThroughputControl",
    "spark.cosmos.throughputControl.targetThroughputThreshold": cosmos_source_throughput_control,
    "spark.cosmos.throughputControl.globalControl.database": cosmos_source_database_name,
    "spark.cosmos.throughputControl.globalControl.container": "ThroughputControl",
    "spark.cosmos.preferredRegionsList": cosmos_region,
}

# Checkpoint location (per source+target db/container) in Lakehouse Files area
checkpoint_location = f"Files/cosmos_migration_checkpoints/{cosmos_source_database_name}/{cosmos_source_container_name}_to_{cosmos_target_database_name}/{cosmos_target_container_name}"
application_name = f"{cosmos_source_database_name}_{cosmos_source_container_name}_to_{cosmos_target_container_name}_"

# Clear stale checkpoint if requested
if clear_checkpoint.lower() == "true":
    import subprocess
    checkpoint_abs = f"/lakehouse/default/{checkpoint_location}"
    try:
        import shutil
        shutil.rmtree(checkpoint_abs)
        print(f"\U0001f5d1\ufe0f Cleared existing checkpoint: {checkpoint_location}")
    except FileNotFoundError:
        print(f"\u2139\ufe0f No existing checkpoint to clear at: {checkpoint_location}")

# Writer configuration (TARGET)
write_cfg = {
    "spark.cosmos.accountEndpoint": cosmos_target_endpoint,
    "spark.cosmos.accountKey": cosmos_target_masterkey,
    "spark.cosmos.applicationName": application_name,
    "spark.cosmos.database": cosmos_target_database_name,
    "spark.cosmos.container": cosmos_target_container_name,
    "spark.cosmos.write.strategy": "ItemOverwrite",
    "checkpointLocation": checkpoint_location,
}

# Read change feed
change_feed_df = (
    spark.readStream
    .format("cosmos.oltp.changeFeed")
    .options(**change_feed_cfg)
    .load()
)

# Preserve source _etag and _ts as audit fields
df_with_audit_fields = change_feed_df.withColumnRenamed("_rawbody", "_origin_rawBody")

# Snapshot source document count BEFORE starting the stream
# Once we've migrated at least this many rows, the initial copy is complete
source_count_df = spark.read.format("cosmos.oltp").options(**{
    "spark.cosmos.accountEndpoint": cosmos_source_endpoint,
    "spark.cosmos.accountKey": cosmos_source_masterkey,
    "spark.cosmos.database": cosmos_source_database_name,
    "spark.cosmos.container": cosmos_source_container_name,
    "spark.cosmos.read.inferSchema.enabled": "false",
    "spark.cosmos.preferredRegionsList": cosmos_region,
}).load()
source_doc_count = source_count_df.count()

# Count existing documents in target to handle resumed migrations
target_count_df = spark.read.format("cosmos.oltp").options(**{
    "spark.cosmos.accountEndpoint": cosmos_target_endpoint,
    "spark.cosmos.accountKey": cosmos_target_masterkey,
    "spark.cosmos.database": cosmos_target_database_name,
    "spark.cosmos.container": cosmos_target_container_name,
    "spark.cosmos.read.inferSchema.enabled": "false",
    "spark.cosmos.preferredRegionsList": cosmos_region,
}).load()
target_existing_count = target_count_df.count()
rows_remaining = max(source_doc_count - target_existing_count, 0)

print(f"Change feed reader initialized for {cosmos_source_database_name}/{cosmos_source_container_name}")
print(f"Source document count: {source_doc_count}")
print(f"Target existing count: {target_existing_count}")
print(f"Rows remaining to migrate: {rows_remaining}")
print(f"Checkpoint location: {checkpoint_location}")

## Step 6: Start Streaming Write

This starts the continuous streaming write from the source change feed to the target container. The stream stops once the total rows migrated reaches the source document count snapshot taken in Step 5, ensuring the initial copy completes without getting stuck on continuously-updated containers.

In [ ]:
import uuid
import time

run_id = str(uuid.uuid4())

# If already fully migrated, skip streaming entirely
if rows_remaining == 0:
    print(f"\u2705 Target already has {target_existing_count} docs (>= {source_doc_count} source). Nothing to migrate.")
    total_rows = 0
else:
    # Start streaming write to target Cosmos DB container
    streaming_query = (
        df_with_audit_fields.writeStream
        .format("cosmos.oltp")
        .queryName(run_id)
        .options(**write_cfg)
        .outputMode("append")
        .start()
    )

    print(f"Streaming migration started for {cosmos_source_container_name} -> {cosmos_target_container_name}")
    print(f"Query ID: {run_id}")
    print(f"Target already has {target_existing_count} docs. Need {rows_remaining} more to reach {source_doc_count}.")

    # Poll until we've migrated at least the remaining rows.
    poll_interval_seconds = 10  # how often to check progress
    total_rows = 0
    initial_wait_max = 60       # max polls to wait for first data (10min at 10s intervals)
    initial_wait_count = 0

    while streaming_query.isActive:
        time.sleep(poll_interval_seconds)
        progress = streaming_query.lastProgress
        if progress:
            num_input_rows = progress.get("numInputRows", 0)
            total_rows += num_input_rows

            if num_input_rows > 0:
                overall = target_existing_count + total_rows
                pct = (overall / source_doc_count * 100) if source_doc_count > 0 else 100
                print(f"  Batch processed {num_input_rows} rows \u2014 overall: {overall}/{source_doc_count} ({pct:.1f}%)")
            else:
                if total_rows == 0:
                    initial_wait_count += 1
                    print(f"  Waiting for first data batch ({initial_wait_count}/{initial_wait_max})...")
                    if initial_wait_count >= initial_wait_max:
                        print(f"\n\u26a0\ufe0f No data seen after {initial_wait_max} polls. Source may be empty or checkpoint already caught up.")
                        streaming_query.stop()
                        break
                else:
                    overall = target_existing_count + total_rows
                    print(f"  Batch idle \u2014 overall: {overall}/{source_doc_count}")

            # Stop once we've migrated at least the remaining rows
            if total_rows >= rows_remaining:
                overall = target_existing_count + total_rows
                print(f"\n\u2705 Migrated {total_rows} rows this session (overall {overall}/{source_doc_count}). Initial copy complete.")
                streaming_query.stop()
                break

        # Also break if the query stopped on its own (error, etc.)
        if not streaming_query.isActive:
            break

    # Check for errors (InterruptedException is expected when we call .stop())
    query_exception = streaming_query.exception()
    if query_exception and "InterruptedException" not in str(query_exception):
        raise Exception(f"Streaming query failed: {query_exception}")

print(f"\n\u2705 Migration complete: {cosmos_source_container_name} -> {cosmos_target_container_name}")
print(f"   Rows migrated this session: {total_rows}")
print(f"   Total in target: {target_existing_count + total_rows}")

# Exit with success status for parent notebook orchestration
notebookutils.notebook.exit(f"SUCCESS: {cosmos_source_container_name} -> {cosmos_target_container_name} ({total_rows} rows this session, {target_existing_count + total_rows} total)")